In [ ]:
# ======================================================================================================
# Cell 12 (add-on, run AFTER Run all AND Cells 9-11 in the same session). PRE-ADMISSION BASELINE LOOKBACK. POST HOC, added 4 Nov 2026.
# Why: the baseline used so far is the lowest creatinine in the first 24 h of the admission (including ED draws). For 70.4% that is not before the procedure day.
# Here the baseline is the MEDIAN of the patient's creatinine values 7 to 365 days BEFORE this admission (outpatient or earlier-admission labs in the labevents table).
# Only the baseline changes; the peak is the same as the primary definition (highest value up to discharge). Aggregates only; counts under SMALL_CELL are masked.
# NEEDS labevents.csv.gz (MIMIC-IV hosp folder; large file, reading takes several minutes). Edit LE_CAND if the path differs.
# ======================================================================================================
say("\n=== CELL 12: pre-admission baseline lookback (median of values 7-365 days before admission) (post hoc) ===")
LE_CAND = [P+"mimic-iv/hosp/labevents.csv.gz", P+"MIMIC 4/labevents.csv.gz", P+"mimic-iv/labevents.csv.gz"]
le_path = next((p for p in LE_CAND if os.path.exists(p)), None)
if not le_path:
    raise SystemExit("CELL 12 needs labevents.csv.gz (MIMIC-IV hosp folder). Edit LE_CAND above to its path.")
subs = set(d.subject_id)
parts = []
for ch in pd.read_csv(le_path, usecols=['subject_id', 'itemid', 'charttime', 'valuenum'], parse_dates=['charttime'], chunksize=5_000_000):
    ch = ch[(ch.itemid == CR_ITEM) & ch.subject_id.isin(subs)].dropna(subset=['valuenum'])
    if len(ch): parts.append(ch[['subject_id', 'charttime', 'valuenum']])
LE = pd.concat(parts, ignore_index=True)
say("Creatinine values found for cohort patients in labevents (all dates):", n_(len(LE)), "| patients:", n_(LE.subject_id.nunique()))
a_ = d[['hadm_id', 'subject_id', 'admittime']].merge(LE, on='subject_id')
gap = a_.admittime - a_.charttime
w_ = a_[(gap >= pd.Timedelta('7D')) & (gap <= pd.Timedelta('365D'))]
bp = w_.groupby('hadm_id').valuenum.median().rename('base_prior')
d = d.drop(columns=[x for x in ['base_prior', 'aki_prior', 'aki_hier'] if x in d.columns]).merge(bp, on='hadm_id', how='left')
d['has_prior'] = d.base_prior.notna().astype(int)
d['aki_prior'] = ((d.peak_all / d.base_prior) >= 1.5).astype(int)
d['base_hier'] = d.base_prior.fillna(d.base24)                      # prior baseline where it exists, else the in-hospital baseline
d['aki_hier'] = ((d.peak_all / d.base_hier) >= 1.5).astype(int)
say("Admissions with a pre-admission baseline (7-365 days before):", n_(d.has_prior.sum()), "of", len(d))
say("By approach (rows; columns 0/1 = has a pre-admission baseline; small cells masked):"); say(cnt(pd.crosstab(d.approach, d.has_prior)).to_string())
if 'base_pre' in d.columns:
    nopre_ = d.base24.notna() & d.base_pre.isna()
    say("Among the admissions with NO creatinine before the procedure day (the 70.4% problem), those that gain a pre-admission baseline:", n_((nopre_ & (d.has_prior == 1)).sum()), "of", n_(nopre_.sum()))
    say("By approach, among those without a pre-procedure-day baseline (rows; columns 0/1 = gains a pre-admission baseline):"); say(cnt(pd.crosstab(d[nopre_].approach, d[nopre_].has_prior)).to_string())
# direct bias diagnostic: is the in-hospital (first 24 h) baseline higher or lower than the patient's own earlier value, and does that differ by approach?
dp = d[d.has_prior == 1].copy(); dp['rr_b'] = dp.base24 / dp.base_prior
say("In-hospital baseline divided by pre-admission baseline (median [IQR]) by approach; 1.00 = identical:")
say(dp.groupby('approach').rr_b.describe()[['25%', '50%', '75%']].round(2).to_string())
say("Count (of admissions with a pre-admission baseline) whose in-hospital baseline is >= 1.25x the pre-admission value (would hide AKI), and <= 0.80x (would create AKI), by approach; small counts masked:")
_t = dp.assign(hi=(dp.rr_b >= 1.25).astype(int), lo=(dp.rr_b <= 0.80).astype(int)).groupby('approach').agg(N=('rr_b', 'size'), hi=('hi', 'sum'), lo=('lo', 'sum'))
_t['hi_pct'] = np.where(_t.hi >= SMALL_CELL, (100 * _t.hi / _t.N).round(1).astype(str) + '%', 'masked'); _t['lo_pct'] = np.where(_t.lo >= SMALL_CELL, (100 * _t.lo / _t.N).round(1).astype(str) + '%', 'masked')
_t['hi'] = _t.hi.map(n_); _t['lo'] = _t.lo.map(n_)
say(_t.to_string())
fit(dp, 'aki', T(), 'P0  Subset with a pre-admission baseline: PRIMARY definition', 'poisson')
fit(dp, 'aki_prior', T(), 'P1  Same subset: pre-admission baseline definition', 'poisson')
fit(d, 'aki_hier', T(), 'P2  All admissions: pre-admission baseline where available, otherwise in-hospital baseline', 'poisson')
say("Reference, primary: RR 1.87 (1.49 to 2.34), 229 events.")


In [ ]:
# ======================================================================================================
# Cell 13 (add-on, run AFTER Cell 12 in the same session). BACK-CALCULATED BASELINE. POST HOC, added 4 Nov 2026.
# Why: a baseline that does not depend on WHEN creatinine was drawn. Standard approach (Acute Disease Quality Initiative): assume a normal eGFR of 75 mL/min/1.73 m2
# and solve the race-free CKD-EPI 2021 equation for the creatinine that gives it, from age and sex only. Known weakness: patients with unrecognised CKD look like AKI,
# so the absolute risk is overstated; what matters here is whether the open-vs-endovascular ratio survives.
# ======================================================================================================
say("\n=== CELL 13: back-calculated baseline creatinine (assumed eGFR 75; CKD-EPI 2021) (post hoc) ===")
fem = d.gender.astype(str).str.upper().str.startswith('F').values
age = d.age_at_admission.astype(float).values
kk = np.where(fem, 0.7, 0.9); aa = np.where(fem, -0.241, -0.302); sx = np.where(fem, 1.012, 1.0)
g0 = 142 * (0.9938 ** age) * sx                                  # eGFR when creatinine/k = 1
d['base_bc'] = kk * np.where(75 > g0, (75 / g0) ** (1 / aa), (g0 / 75) ** (1 / 1.2))
chk_ = 142 * np.where(d.base_bc / kk < 1, (d.base_bc / kk) ** aa, (d.base_bc / kk) ** -1.2) * (0.9938 ** age) * sx
say("CHECK: eGFR recomputed from the back-calculated creatinine (must be 75.0 for every admission): min", round(float(np.nanmin(chk_)), 2), "max", round(float(np.nanmax(chk_)), 2))
say("Gender values seen (female = starts with F):", mask_dict(d.gender.value_counts()))
say("Back-calculated baseline, mg/dL (median [IQR]) by sex:"); say(d.assign(sex=np.where(fem, 'female', 'male')).groupby('sex').base_bc.describe()[['25%', '50%', '75%']].round(2).to_string())
d['aki_bc'] = ((d.peak_all / d.base_bc) >= 1.5).astype(int)                       # baseline from age and sex only
d['base_lo'] = np.fmin(d.base24, d.base_bc)                                      # lower of measured and back-calculated
d['aki_bclo'] = ((d.peak_all / d.base_lo) >= 1.5).astype(int)
say("Peak creatinine missing for", n_(d.peak_all.isna().sum()), "admissions (counted as no AKI in these definitions).")
say("Crude AKI, back-calculated baseline: open", f"{d[d.approach=='open'].aki_bc.mean():.1%}", "| endovascular", f"{d[d.approach=='endovascular'].aki_bc.mean():.1%}",
    "| events", n_(d.aki_bc.sum()))
say("Crude AKI, lower of measured and back-calculated baseline: open", f"{d[d.approach=='open'].aki_bclo.mean():.1%}", "| endovascular", f"{d[d.approach=='endovascular'].aki_bclo.mean():.1%}",
    "| events", n_(d.aki_bclo.sum()))
say("Agreement with the primary definition (rows = primary AKI 0/1; columns = back-calculated AKI 0/1; small cells masked):"); say(cnt(pd.crosstab(d.aki, d.aki_bc)).to_string())
fit(d, 'aki_bc', T(), 'B1  AKI = peak >= 1.5x back-calculated baseline (no timing dependence)', 'poisson')
fit(d, 'aki_bclo', T(), 'B2  AKI = peak >= 1.5x the LOWER of measured and back-calculated baseline', 'poisson')
fit(d[d.rupture == 0], 'aki_bc', [t for t in T() if t != 'rupture'], 'B3  As B1, excluding ruptured repairs', 'poisson')
say("Reference, primary: RR 1.87 (1.49 to 2.34), 229 events. Read B1 with care: it labels anyone whose peak is about 1.5 mg/dL or more as AKI, so unrecognised CKD counts.")


In [ ]:
# ======================================================================================================
# Cell 14 (add-on, run AFTER Cell 13 in the same session). MORE PRE-PROCEDURE COVARIATES AND A BENCHMARK FOR THE E-VALUE. POST HOC, added 4 Nov 2026.
# Why: the primary model has age, sex, location, era, rupture, urgency and the Charlson index only. Hypertension, coronary disease, obesity and tobacco use are
# standard risk factors for AKI after aortic surgery and are not (or only partly) in the Charlson index. Atrial fibrillation is NOT added: it is often post-operative.
# Caveat: MIMIC diagnosis codes are assigned at discharge, so some coded diagnoses may arise during the stay (this is also true for the Charlson index).
# ======================================================================================================
say("\n=== CELL 14: additional covariates and E-value benchmark (post hoc) ===")
if 'dx' not in globals():
    dx = pd.read_csv(F['dx'], usecols=['hadm_id', 'icd_code', 'icd_version'], dtype={'icd_code': str}); dx = dx[dx.hadm_id.isin(ids)]
COMORB = {'htn':     (('401', '402', '403', '404', '405'), ('I10', 'I11', 'I12', 'I13', 'I15', 'I16')),
          'cad':     (('410', '411', '412', '413', '414'), ('I20', 'I21', 'I22', 'I24', 'I25')),
          'obesity': (('2780',), ('E66',)),
          'tobacco': (('3051', 'V1582'), ('F17', 'Z720', 'Z87891'))}
for nm_, (p9, p10) in COMORB.items():
    m_ = ((dx.icd_version == 9) & dx.icd_code.str.startswith(p9)) | ((dx.icd_version == 10) & dx.icd_code.str.startswith(p10))
    d[nm_] = d.hadm_id.isin(set(dx[m_].hadm_id)).astype(int)
say("Prevalence by approach (rows = approach; columns 0/1; small cells masked):")
for nm_ in COMORB:
    say(f"  {nm_}:"); say(cnt(pd.crosstab(d.approach, d[nm_])).to_string())
EX = list(COMORB)
mm = fit(d, 'aki', T(EX), 'C1  Primary covariates + hypertension, coronary disease, obesity, tobacco use', 'poisson')
fit(d[d.rupture == 0], 'aki', [t for t in T(EX) if t != 'rupture'], 'C2  As C1, excluding ruptured repairs', 'poisson')
# benchmark for the E-value: how strong are the covariates we DID measure?
if mm is not None:
    say("Benchmark for the E-value (3.14): adjusted risk ratios for AKI of the measured covariates in C1, and open/endovascular prevalence ratios:")
    rows_ = []
    for nm_ in ['rupture'] + EX:
        k_ = [x for x in mm.params.index if x == nm_][0]; lo_, hi_ = np.exp(mm.conf_int().loc[k_])
        co, ce = int(d[d.approach == 'open'][nm_].sum()), int(d[d.approach == 'endovascular'][nm_].sum())
        no_, ne_ = int((d.approach == 'open').sum()), int((d.approach == 'endovascular').sum())
        ok_ = all(c_ >= SMALL_CELL and (n__ - c_) >= SMALL_CELL for c_, n__ in [(co, no_), (ce, ne_)])
        rows_.append((nm_, round(float(np.exp(mm.params[k_])), 2), f"{lo_:.2f} to {hi_:.2f}",
                      f"{100*co/no_:.1f}%" if ok_ else 'masked', f"{100*ce/ne_:.1f}%" if ok_ else 'masked', round((co/no_)/(ce/ne_), 2) if ok_ else 'masked'))
    say(pd.DataFrame(rows_, columns=['covariate', 'RR for AKI', '95% CI', 'prevalence open', 'prevalence endovascular', 'prevalence ratio']).to_string(index=False))
    say("An unmeasured confounder would have to be associated with AKI AND with repair type by risk ratios of at least 3.1 each to explain the primary estimate.")
say("Reference, primary: RR 1.87 (1.49 to 2.34), 229 events.")
